In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping


d:\Mohamed Refaat\Anaconda_2\envs\tf_final\lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:

# =========================
# 1️⃣ Load data
# =========================
df = pd.read_csv(r'D:\Mohamed Refaat\DEPI\Project implemention\Intelligent-Support-Ticket\Project Implementation\Data\clean_data.csv')
X = df['initial_message']
y = df['customer_sentiment']

# =========================
# 2️⃣ Split data
# =========================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =========================
# 3️⃣ Encode labels
# =========================
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [5]:

# =========================
# 4️⃣ Build model with NNLM
# =========================
nnlm_embedding = hub.KerasLayer(
    "https://tfhub.dev/google/nnlm-en-dim128/2",
    input_shape=[], 
    dtype=tf.string,
    trainable=False,
    name="nnlm_embedding"
)

KeyboardInterrupt: 

In [ ]:


text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
x = nnlm_embedding(text_input)
x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(5, activation='softmax')(x)


In [ ]:

model = tf.keras.Model(inputs=text_input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:



# =========================
# 5️⃣ Setup Early Stopping
# =========================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)



In [ ]:
# =========================
# 6️⃣ Train model
# =========================
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=32,
    callbacks=[early_stop]
)


In [ ]:

# =========================
# 7️⃣ Evaluate model
# =========================
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")